In [3]:
import pandas as pd
import numpy as np
import random
import os

# 1. Load the Raw Data
# Make sure your CSV is in the data/raw folder!
file_path = 'D:\\Desktop\\BootCamp_Hackathon\\KaggleV2-May-2016.csv' 
df = pd.read_csv(file_path)

print(f"Original data shape: {df.shape}")

# 2. Clean up Column Names (make them easier to type)
df = df.rename(columns={
    'Hipertension': 'Hypertension',
    'Handcap': 'Handicap',
    'No-show': 'NoShow'
})

# 3. Target Variable: Convert 'Yes'/'No' to 1 and 0
# Yes = They NO-SHOWED (1), No = They showed up (0)
df['NoShow'] = df['NoShow'].map({'Yes': 1, 'No': 0})

# 4. Handle Dates & Calculate "Lead Time" (Crucial Feature!)
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay']).dt.tz_localize(None).dt.normalize()
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay']).dt.tz_localize(None).dt.normalize()

# LeadDays = How many days between booking and the actual appointment
df['LeadDays'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days

# Fix data entry errors (LeadDays cannot be negative)
df = df[df['LeadDays'] >= 0] 

# Calculate Day of the week (0 = Monday, 6 = Sunday)
df['DayOfWeek'] = df['AppointmentDay'].dt.dayofweek

# 5. Add the Synthetic "Specialty" (So we can match with Waitlist later)
def assign_specialty(row):
    if row['Age'] < 14:
        return 'Pediatrics'
    elif row['Hypertension'] == 1 or row['Diabetes'] == 1:
        return 'Cardiology'
    elif row['Age'] > 60:
        return 'Internal Medicine'
    else:
        return random.choice(['General Practice', 'Orthopedics', 'Neurology'])

# Apply the function to create a new column
np.random.seed(42)
random.seed(42)
df['Specialty'] = df.apply(assign_specialty, axis=1)

# 6. Drop columns we don't need for the ML model
columns_to_drop = ['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay', 'Neighbourhood']
df_processed = df.drop(columns=columns_to_drop)

# 7. Save to the Processed Folder
os.makedirs('D:\\Desktop\\BootCamp_Hackathon\\data\\processed', exist_ok=True)
df_processed.to_csv('D:\\Desktop\\BootCamp_Hackathon\\data\\processed/cleaned_appointments.csv', index=False)

print(f"Processed data shape: {df_processed.shape}")
print("✅ Data cleaned, features added, and saved to 'data/processed/cleaned_appointments.csv'")

# Take a quick peek at your new data
df_processed.head()

Original data shape: (110527, 14)
Processed data shape: (110522, 12)
✅ Data cleaned, features added, and saved to 'data/processed/cleaned_appointments.csv'


,Gender,Age,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,NoShow,LeadDays,DayOfWeek,Specialty
0,F,62,0,1,0,0,0,0,0,0,4,Cardiology
1,M,56,0,0,0,0,0,0,0,0,4,Neurology
2,F,62,0,0,0,0,0,0,0,0,4,Internal Medicine
3,F,8,0,0,0,0,0,0,0,0,4,Pediatrics
4,F,56,0,1,1,0,0,0,0,0,4,Cardiology
